<a href="https://colab.research.google.com/github/dontasksteven/IS4487/blob/main/Assignments/assignment_12_PetersonStevenl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 12: Predicting Hotel Booking Cancellations  
## Models: Naïve Bayes, Support Vector Machine (SVM), and Neural Network

**Objectives:**
- Understand how to use classification models (Naïve Bayes, SVM, Neural Networks) to predict hotel cancellations.
- Compare models in terms of accuracy, complexity, and business relevance.
- Interpret and communicate model results from a business perspective.

## Business Scenario

You work as a data analyst for a hospitality group that manages both **Resort** and **City Hotels**. One major challenge in operations is the unpredictability of **booking cancellations**, which affects staffing, inventory, and revenue planning.

You’ve been asked to use historical booking data to predict whether a future booking will be canceled. Your insights will help management plan more effectively.


Your task is to:
1. Build and evaluate three models: Naïve Bayes, SVM, and Neural Network.
2. Compare performance.
3. Recommend which model is best suited for the business needs.

<a href="https://colab.research.google.com/github/vandanara/UofUtah_IS4487/blob/main/Assignments/assignment_12_bayes_svm_neural.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>


## Dataset Description: Hotel Bookings

This dataset contains booking information for two types of hotels: a **city hotel** and a **resort hotel**. Each record corresponds to a single booking and includes various details about the reservation, customer demographics, booking source, and whether the booking was canceled.

**Source**: [GitHub - TidyTuesday: Hotel Bookings](https://github.com/rfordatascience/tidytuesday/blob/master/data/2020/2020-02-11/readme.md)

### Key Use Cases
- Understand customer booking behavior
- Explore factors related to cancellations
- Segment guests based on booking characteristics
- Compare city vs. resort hotel performance

### Data Dictionary

| Variable | Type | Description |
|----------|------|-------------|
| `hotel` | character | Hotel type: City or Resort |
| `is_canceled` | integer | 1 = Canceled, 0 = Not Canceled |
| `lead_time` | integer | Days between booking and arrival |
| `arrival_date_year` | integer | Year of arrival |
| `arrival_date_month` | character | Month of arrival |
| `stays_in_weekend_nights` | integer | Nights stayed on weekends |
| `stays_in_week_nights` | integer | Nights stayed on weekdays |
| `adults` | integer | Number of adults |
| `children` | integer | Number of children |
| `babies` | integer | Number of babies |
| `meal` | character | Type of meal booked |
| `country` | character | Country code of origin |
| `market_segment` | character | Booking source (e.g., Direct, Online TA) |
| `distribution_channel` | character | Booking channel used |
| `is_repeated_guest` | integer | 1 = Repeated guest, 0 = New guest |
| `previous_cancellations` | integer | Past booking cancellations |
| `previous_bookings_not_canceled` | integer | Past bookings not canceled |
| `reserved_room_type` | character | Initially reserved room type |
| `assigned_room_type` | character | Room type assigned at check-in |
| `booking_changes` | integer | Number of booking modifications |
| `deposit_type` | character | Deposit type (No Deposit, Non-Refund, etc.) |
| `agent` | character | Agent ID who made the booking |
| `company` | character | Company ID (if booking through company) |
| `days_in_waiting_list` | integer | Days on the waiting list |
| `customer_type` | character | Booking type: Contract, Transient, etc. |
| `adr` | float | Average Daily Rate (price per night) |
| `required_car_parking_spaces` | integer | Requested parking spots |
| `total_of_special_requests` | integer | Number of special requests made |
| `reservation_status` | character | Final status (Canceled, No-Show, Check-Out) |
| `reservation_status_date` | date | Date of the last status update |

This dataset is ideal for classification, segmentation, and trend analysis exercises.


## 1. Load and Prepare the Hotel Booking Dataset

**Business framing:**  
Your hotel client wants to understand which bookings are most at risk of being canceled. But before modeling, your job is to prepare the data to ensure clean and reliable input.

### Do the following:
- Load the `hotels.csv` file from https://raw.githubusercontent.com/vandanara/UofUtah_IS4487/refs/heads/main/DataSets/hotels.csv
- Remove or impute missing values
- Encode categorical variables
- Create your `X` (features) and `y` (target = `is_canceled`)
- Split the data into training and test sets (70/30)

### In Your Response:
1. How many total rows and columns are in the dataset?
2. What types of features (categorical, numerical) are included?
3. What steps did you take to clean or prepare the data?


In [10]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
url = "https://raw.githubusercontent.com/vandanara/UofUtah_IS4487/refs/heads/main/DataSets/hotels.csv"

df=pd.read_csv(url)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 32 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           119390 non-null  object 
 1   is_canceled                     119390 non-null  int64  
 2   lead_time                       119390 non-null  int64  
 3   arrival_date_year               119390 non-null  int64  
 4   arrival_date_month              119390 non-null  object 
 5   arrival_date_week_number        119390 non-null  int64  
 6   arrival_date_day_of_month       119390 non-null  int64  
 7   stays_in_weekend_nights         119390 non-null  int64  
 8   stays_in_week_nights            119390 non-null  int64  
 9   adults                          119390 non-null  int64  
 10  children                        119386 non-null  float64
 11  babies                          119390 non-null  int64  
 12  meal            

In [11]:
df = df.drop(columns=['arrival_date_year', 'arrival_date_month', 'arrival_date_week_number', 'arrival_date_day_of_month', 'distribution_channel', 'deposit_type', 'adr', 'agent', 'company', 'market_segment'], errors='ignore')

In [12]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Define the URL (ensure it's available or loaded from previous cell)
url = "https://raw.githubusercontent.com/vandanara/UofUtah_IS4487/refs/heads/main/DataSets/hotels.csv"

# Reload the original DataFrame to ensure a fresh state for processing
df = pd.read_csv(url)

# Apply the initial column drops as intended by cell I5kiHgpECRbX
df = df.drop(columns=['arrival_date_year', 'arrival_date_month', 'arrival_date_week_number', 'arrival_date_day_of_month', 'distribution_channel', 'deposit_type', 'adr', 'agent', 'company', 'market_segment'], errors='ignore')

# Handle missing values
df['children'] = df['children'].fillna(0)
df['country'] = df['country'].fillna(df['country'].mode()[0])

# Drop columns that are either direct leakage (reservation_status, reservation_status_date)
df.drop(columns=['reservation_status', 'reservation_status_date'], inplace=True, errors='ignore')

# Identify categorical columns for encoding
categorical_cols = df.select_dtypes(include='object').columns.tolist()

# Apply One-Hot Encoding
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# Prepare features (X) and target (y)
X = df.drop('is_canceled', axis=1)
y = df['is_canceled']

# Split the data into training and test sets (70/30)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print(f"Original dataset shape: {pd.read_csv(url).shape}")
print(f"Processed dataset shape (after encoding and dropping): {df.shape}")
print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test: {y_test.shape}")


Original dataset shape: (119390, 32)
Processed dataset shape (after encoding and dropping): (119390, 218)
Shape of X_train: (83573, 217)
Shape of X_test: (35817, 217)
Shape of y_train: (83573,)
Shape of y_test: (35817,)


In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Columns: 218 entries, is_canceled to customer_type_Transient-Party
dtypes: bool(204), float64(1), int64(13)
memory usage: 36.0 MB


### ✍️ Your Response: 🔧
1. There are 119390 rows and 218 columns after we cleaned our dataset.

2. It appears that most of our data types are binary (bool). Other than that we have numerical data types from float64 and int64.

3. The first step in cleaning our data, after importing, was to drop unnecessary columns and fill in missing data. From there we identified our categorical variables and applied one-hot encoding to those. Finally, we created a 70/30 split in our data for future analysis.

## 2. Build a Naïve Bayes Model

**Business framing:**  
Naïve Bayes is a quick, baseline model often used for early testing or simple classification problems.

### Do the following:
- Train a Naïve Bayes classifier on your training data
- Use it to predict on your test data
- Print a classification report and confusion matrix

### In Your Response:
1. How well does the model perform?  And what metric is best used to judge the performance?
2. Where might this model be useful for the hotel (e.g. real-time alerts, operational decisions)?


In [14]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report, confusion_matrix

# Initialize and train the Naïve Bayes classifier
nb_model = GaussianNB()
nb_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred_nb = nb_model.predict(X_test)

# Print classification report and confusion matrix
print("Naïve Bayes Classification Report:")
print(classification_report(y_test, y_pred_nb))
print("\nNaïve Bayes Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_nb))
print("Accuracy:", accuracy_score(y_test, y_pred_nb))

Naïve Bayes Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.27      0.41     22478
           1       0.43      0.93      0.59     13339

    accuracy                           0.52     35817
   macro avg       0.65      0.60      0.50     35817
weighted avg       0.70      0.52      0.48     35817


Naïve Bayes Confusion Matrix:
[[ 6112 16366]
 [  992 12347]]
Accuracy: 0.5153697964653656


### ✍️ Your Response: 🔧
1. We have an accuracy score of .52 so overall our model is not very accurate and we likely would have similar accuracy if we were just guessing. In addition to accuracy we should look at recall. For values that fall into class 1 we catch nearly all cases (93%), however with a precision of .43 we have a lot of "false positives."

2. This model would be most useful if we are trying to identify class 1 cases, in our case this would be problems or high risk guests. This would make our model useful for predicting complaints and identifying dissatisfied guests.


## 3. Build a Support Vector Machine (SVM) Model

**Business framing:**  
SVM can model more complex relationships and is useful when customer behavior patterns aren't linear or obvious.

### Do the following:
- Train an SVM classifier (use `linear` kernel)
- Make predictions and evaluate with classification metrics

### In Your Response:
1. How well does the model perform?  And what metric is best used to judge the performance?
2. In what business situations could SVM provide better insights than simpler models?


In [15]:
# SVM for the hotel dataset (uses your existing X_train, X_test, y_train, y_test)
from sklearn.svm import LinearSVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import time
import numpy as np

# Ensure we have numeric floats (LinearSVC requires float)
# X_train / X_test might be DataFrames with bool/int types after one-hot encoding.
X_train_float = X_train.astype(float)
X_test_float  = X_test.astype(float)

# Scale all features (0/1 binary + numeric columns). Scaling binary is OK.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_float)
X_test_scaled  = scaler.transform(X_test_float)

# Choose dual parameter for speed:
# As recommended by scikit-learn, set dual=False when n_samples > n_features
n_samples, n_features = X_train_scaled.shape
dual_setting = False if n_samples > n_features else True

# Initialize LinearSVC (fast linear SVM)
svm_model = LinearSVC(C=1.0, max_iter=10000, dual=dual_setting, random_state=42)

# Train and time it
start = time.time()
svm_model.fit(X_train_scaled, y_train)
end = time.time()
print(f"Training time: {end - start:.2f} seconds")

# Predict and evaluate
y_pred_svm = svm_model.predict(X_test_scaled)

print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_svm))
print("\nClassification Report:\n", classification_report(y_test, y_pred_svm))
print("Accuracy:", accuracy_score(y_test, y_pred_svm))


Training time: 21.05 seconds
Confusion Matrix:
 [[19756  2722]
 [ 4782  8557]]

Classification Report:
               precision    recall  f1-score   support

           0       0.81      0.88      0.84     22478
           1       0.76      0.64      0.70     13339

    accuracy                           0.79     35817
   macro avg       0.78      0.76      0.77     35817
weighted avg       0.79      0.79      0.79     35817

Accuracy: 0.7904905491805567


### ✍️ Your Response: 🔧
1. With an accuracy of .79 this is a major improvement compared to our Naive Bayes model. Our model predicts class 0 better than it does 1. Our recall and precision results are relatively comparable so we are catching a good chunk of not high risk guests and we haven't misclassed too many of them in turn.

2. Our SVM model would be useful for predicting cancellation risk. Despite this being similar to our Naive Bayes model, our SVM performs better as we don't make the assumption that our values are independent.

## 4. Build a Neural Network Model

**Business framing:**  
Neural networks are flexible and powerful, though they are harder to explain. They may work well when subtle patterns exist in the data.

### Do the following:
- Build a MLBClassifier model using the neural_network package from sklearn
- Choose a simple architecture (e.g., 2 hidden layers)
- Evaluate accuracy and performance

### In Your Response:
1. How does this model compare to the others?
2. Would the business be comfortable using a “black box” model like this? Why or why not?


In [16]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import time

# Initialize the MLPClassifier with a simple architecture (2 hidden layers)
# Using X_train_scaled and X_test_scaled from previous steps
mlp_model = MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=500, random_state=42, verbose=True)

# Train the model and time it
print("\nTraining Neural Network model...")
start = time.time()
mlp_model.fit(X_train_scaled, y_train)
end = time.time()
print(f"Training time: {end - start:.2f} seconds")

# Make predictions on the test set
y_pred_mlp = mlp_model.predict(X_test_scaled)

# Print classification report and confusion matrix
print("\nNeural Network Classification Report:")
print(classification_report(y_test, y_pred_mlp))
print("\nNeural Network Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_mlp))
print("Accuracy:", accuracy_score(y_test, y_pred_mlp))


Training Neural Network model...
Iteration 1, loss = 0.43812723
Iteration 2, loss = 0.38702780
Iteration 3, loss = 0.37621459
Iteration 4, loss = 0.36852690
Iteration 5, loss = 0.36364293
Iteration 6, loss = 0.35888611
Iteration 7, loss = 0.35510956
Iteration 8, loss = 0.35205258
Iteration 9, loss = 0.34913747
Iteration 10, loss = 0.34700510
Iteration 11, loss = 0.34449483
Iteration 12, loss = 0.34211267
Iteration 13, loss = 0.33951562
Iteration 14, loss = 0.33815769
Iteration 15, loss = 0.33539792
Iteration 16, loss = 0.33414932
Iteration 17, loss = 0.33240060
Iteration 18, loss = 0.33113277
Iteration 19, loss = 0.32860538
Iteration 20, loss = 0.32758050
Iteration 21, loss = 0.32621554
Iteration 22, loss = 0.32468058
Iteration 23, loss = 0.32402921
Iteration 24, loss = 0.32274844
Iteration 25, loss = 0.32140188
Iteration 26, loss = 0.31991627
Iteration 27, loss = 0.31899981
Iteration 28, loss = 0.31824656
Iteration 29, loss = 0.31680184
Iteration 30, loss = 0.31513971
Iteration 31, l

### ✍️ Your Response: 🔧
1. Our Neural Network model performed the best compared to our other models. It had an accuracy of .83 (rounded) compared to our second highest of .79 from our SVM model; this is a marginal improvement. Furthermore, our SVM model was best at predicting class 0 but our neural network even slightly out performs that.

2. If a business just wants to know whether someone may cancel or not and doesn't care to know the reasoning why then the neural network is perfect. However, if a business is looking to understand the reasons and factors why one may cancel then the SVM model may be better despite having a slightly smaller accuracy.


## 5. Compare All Three Models

### Do the following:
- Print and compare the accuracy of Naïve Bayes, SVM, and Neural Network models
- Summarize which model performed best

### In Your Response:
1. Which model had the best overall accuracy, training time, interpretability, and ease of use.
2. Would you recommend this model for deployment, and why?


In [17]:
print("Accuracy of Naive Bayes:", accuracy_score(y_test, y_pred_nb))
print("Accuracy of SVM:", accuracy_score(y_test, y_pred_svm))
print("Accuracy of Neural Network:", accuracy_score(y_test, y_pred_mlp))

Accuracy of Naive Bayes: 0.5153697964653656
Accuracy of SVM: 0.7904905491805567
Accuracy of Neural Network: 0.8282379875478125


### ✍️ Your Response: 🔧
1. Overall our Neural Network has the greatest accuracy compared to the other models, despite this there lacks interpretability and the training time took by far the longest (nearly 12 minutes). The best overall model considering all of the factors in my opinion is our SVM model. The accuracy wasn't too far off of our neural network, the training time was relatively short after applying some scaling, the interpretability is actually present and we can see the reasoning. Due to these factors I believe that it is the best overall model

2. Yes I would recommend it. Having the transparency that the neural network lacks and nearly the same accuracy alone makes it worth it to use over the neural network. Furthermore, it is better than the Naive Bayes as we don't assume independence and our accuracy is much higher.


## 6. Final Business Recommendation

### In Your Response:
1. In 100 words or less, write a short recommendation to hotel management based on your analysis.

Possible info to include:
- Which model do you recommend implementing?
- What business problem does it help solve?
- Are there any risks or limitations?
- What additional data might improve the results in the future?
2. How does this relate to your customized learning outcome you created in canvas?


### ✍️ Your Response: 🔧
1. As we are looking to predict cancellation risk I recommend utilizing our SVM model. The accuracy of the model is nearly as good as the more complex neural network and its computation time is much shorter. Furthermore, by utilizing the SVM model we retain data transparency and keep the ability to figure out the "why" in the data.

2. Looking at risk and deciding the overall feasibility of different models directly relates to my goal of Blockchain technology integration. By being able to identify the best model to implement the corporation's goals specifically to identify risk aligns nearly perfectly with my learning outcome.


## Submission Instructions
✅ Checklist:
- All code cells run without error
- All markdown responses are complete
- Submit on Canvas as instructed

In [18]:
!jupyter nbconvert --to html "assignment_12_PetersonSteven.ipynb"

[NbConvertApp] Converting notebook assignment_12_PetersonSteven.ipynb to html
[NbConvertApp] Writing 340849 bytes to assignment_12_PetersonSteven.html
